In [ ]:
0.13.52

In [ ]:
import polars as pl
from datetime import datetime, date

def QA_fetch_get_stock_min(code, start, end, frequence='1min', ip=None, port=None):
    ip, port = get_mainmarket_ip(ip, port)
    api = TdxHq_API()
    type_ = ''
    
    start_date = str(start)[:10]
    today_ = date.today()
    lens = QA_util_get_trade_gap(start_date, today_)
    
    # 频率映射与长度计算
    freq_map = {
        ('5', '5m', '5min', 'five'): (0, '5min', 48),
        ('1', '1m', '1min', 'one'): (8, '1min', 240),
        ('15', '15m', '15min', 'fifteen'): (1, '15min', 16),
        ('30', '30m', '30min', 'half'): (2, '30min', 8),
        ('60', '60m', '60min', '1h'): (3, '60min', 4),
    }
    
    str_freq = str(frequence)
    for keys, (f, t, multiplier) in freq_map.items():
        if str_freq in keys:
            frequence, type_, lens = f, t, multiplier * lens
            break
            
    if lens > 20800:
        lens = 20800
        
    # 分批获取数据并使用 polars 拼接
    frames = []
    with api.connect(ip, port):
        for i in range(int(lens / 800) + 1):
            offset = (int(lens / 800) - i) * 800
            df = api.to_df(api.get_security_bars(
                frequence, 
                _select_market_code(str(code)), 
                str(code), 
                offset, 
                800
            ))
            # 如果 to_df 返回的是 pandas dataframe，先转为 polars
            if not isinstance(df, pl.DataFrame):
                df = pl.from_pandas(df)
            frames.append(df)
            
        data = pl.concat(frames, how="vertical")
        
    # 向量化处理列，替代 pandas 的 apply 和 assign
    data = (
        data
        .drop(['year', 'month', 'day', 'hour', 'minute'])
        .with_columns([
            pl.col('datetime').str.to_datetime().alias('datetime'),
            pl.lit(str(code)).alias('code'),
            pl.col('datetime').str.slice(0, 10).alias('date'),
            pl.col('datetime').map_elements(QA_util_date_stamp, return_dtype=pl.Int64).alias('date_stamp'),
            pl.col('datetime').map_elements(QA_util_time_stamp, return_dtype=pl.Int64).alias('time_stamp'),
            pl.lit(type_).alias('type'),
        ])
        # 替代 set_index 和 [start:end] 切片，直接用布尔过滤
        .filter(
            (pl.col('datetime') >= start) & 
            (pl.col('datetime') <= end)
        )
        # 将 datetime 转回字符串
        .with_columns(
            pl.col('datetime').cast(pl.Utf8).alias('datetime')
        )
    )
    
    return data

In [31]:
from QUANTAXIS.QAFetch.QATdx import get_mainmarket_ip

In [32]:
from pytdx.hq import TdxHq_API

In [35]:
from QUANTAXIS.QAUtil.QADate_trade import QA_util_get_trade_gap
from QUANTAXIS.QAUtil.QADate import QA_util_date_stamp, QA_util_time_stamp

In [34]:
from QUANTAXIS.QAFetch.base import _select_market_code

In [10]:
QA_fetch_get_stock_min('000001','2026-06-01','2026-07-17','1min',None,None)

<ipython-input-9-22fdc85546c4>:56: DeprecationWarning: `fmt` is deprecated as an argument to `strptime`; use `format` instead.
  pl.col('datetime').str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False).alias('datetime'),
<ipython-input-9-22fdc85546c4>:69: DeprecationWarning: `fmt` is deprecated as an argument to `strptime`; use `format` instead.
  (pl.col('datetime') >= pl.lit(start).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False)) &
<ipython-input-9-22fdc85546c4>:70: DeprecationWarning: `fmt` is deprecated as an argument to `strptime`; use `format` instead.
  (pl.col('datetime') <= pl.lit(end).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False))


open,close,high,low,vol,amount,datetime,code,type,date,date_stamp,time_stamp
f64,f64,f64,f64,f64,f64,str,str,str,str,f64,f64


In [52]:
frequence = '1min'
code = '000001'
offset = 1600
start = '2026-06-01'
end = '2026-07-17'

In [17]:
api = TdxHq_API()
df = api.to_df(api.get_security_bars(
    frequence, 
    _select_market_code(str(code)), 
    str(code), 
    offset, 
    800
))

In [20]:
_select_market_code(str(code))

0

In [59]:
ip, port = get_mainmarket_ip(None, None)
api = TdxHq_API()
type_ = ''

start_time = time.time()

start_date = str(start)[:10]
today_ = date.today()
lens = QA_util_get_trade_gap(start_date, today_)

# 频率映射与长度计算
freq_map = {
    ('5', '5m', '5min', 'five'): (0, '5min', 48),
    ('1', '1m', '1min', 'one'): (8, '1min', 240),
    ('15', '15m', '15min', 'fifteen'): (1, '15min', 16),
    ('30', '30m', '30min', 'half'): (2, '30min', 8),
    ('60', '60m', '60min', '1h'): (3, '60min', 4),
}

str_freq = str(frequence)
for keys, (f, t, multiplier) in freq_map.items():
    if str_freq in keys:
        frequence, type_, lens = f, t, multiplier * lens
        break

if lens > 20800:
    lens = 20800

# 分批获取数据并使用 polars 拼接
frames = []
with api.connect(ip, port):
    for i in range(int(lens / 800) + 1):
        offset = (int(lens / 800) - i) * 800
        df = api.to_df(api.get_security_bars(
            frequence, 
            _select_market_code(str(code)), 
            str(code), 
            offset, 
            800
        ))
        # 兼容 API 返回 Pandas 的情况
        if not isinstance(df, pl.DataFrame):
            df = pl.from_pandas(df)
        frames.append(df)

pl_data = pl.concat(frames, how="vertical")

end_time = time.time()
print(f"策略运行耗时: {end_time - start_time:.4f} 秒")

策略运行耗时: 0.2583 秒


In [25]:
lens

8400

In [41]:
pl_data

open,close,high,low,vol,amount,year,month,day,hour,minute,datetime
f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,str
10.6,10.62,10.62,10.59,640300.0,6.78986e6,2026,7,14,13,41,"""2026-07-14 13:…"
10.62,10.61,10.62,10.6,346300.0,3.673281e6,2026,7,14,13,42,"""2026-07-14 13:…"
10.61,10.62,10.62,10.6,316800.0,3.361279e6,2026,7,14,13,43,"""2026-07-14 13:…"
10.62,10.62,10.62,10.61,343300.0,3.645397e6,2026,7,14,13,44,"""2026-07-14 13:…"
10.62,10.62,10.62,10.61,228700.0,2.42697e6,2026,7,14,13,45,"""2026-07-14 13:…"
10.62,10.61,10.62,10.61,107100.0,1.136704e6,2026,7,14,13,46,"""2026-07-14 13:…"
10.62,10.63,10.63,10.61,408600.0,4.339149e6,2026,7,14,13,47,"""2026-07-14 13:…"
10.63,10.63,10.63,10.62,318600.0,3.384684e6,2026,7,14,13,48,"""2026-07-14 13:…"
10.63,10.63,10.64,10.62,655900.0,6.970896e6,2026,7,14,13,49,"""2026-07-14 13:…"


In [58]:
import datetime
import pandas as pd
import time

start_time = time.time()
# start_time = datetime.now()

ip, port = get_mainmarket_ip(None, None)
api = TdxHq_API()
type_ = ''
start_date = str(start)[0:10]
today_ = datetime.date.today()
lens = QA_util_get_trade_gap(start_date, today_)
if str(frequence) in ['5', '5m', '5min', 'five']:
    frequence, type_ = 0, '5min'
    lens = 48 * lens
elif str(frequence) in ['1', '1m', '1min', 'one']:
    frequence, type_ = 8, '1min'
    lens = 240 * lens
elif str(frequence) in ['15', '15m', '15min', 'fifteen']:
    frequence, type_ = 1, '15min'
    lens = 16 * lens
elif str(frequence) in ['30', '30m', '30min', 'half']:
    frequence, type_ = 2, '30min'
    lens = 8 * lens
elif str(frequence) in ['60', '60m', '60min', '1h']:
    frequence, type_ = 3, '60min'
    lens = 4 * lens
if lens > 20800:
    lens = 20800
with api.connect(ip, port):

    data = pd.concat(
        [api.to_df(
            api.get_security_bars(
                frequence, _select_market_code(
                    str(code)),
                str(code),
                (int(lens / 800) - i) * 800, 800)) for i
         in range(int(lens / 800) + 1)], axis=0, sort=False)
    data = data \
        .drop(['year', 'month', 'day', 'hour', 'minute'], axis=1,
              inplace=False) \
        .assign(datetime=pd.to_datetime(data['datetime'], utc=False),
                code=str(code),
                date=data['datetime'].apply(lambda x: str(x)[0:10]),
                date_stamp=data['datetime'].apply(
            lambda x: QA_util_date_stamp(x)),
            time_stamp=data['datetime'].apply(
            lambda x: QA_util_time_stamp(x)),
            type=type_).set_index('datetime', drop=False,
                                  inplace=False)[start:end]
pd_data = data.assign(datetime=data['datetime'].apply(lambda x: str(x)))

end_time = time.time()
print(f"策略运行耗时: {end_time - start_time:.4f} 秒")

# end_time = datetime.now()
# duration = end_time - start_time
# print(f"策略运行耗时: {duration}")  # 输出类似: 0:00:05.123456

策略运行耗时: 0.3036 秒


In [60]:
pd_data.head()

,open,close,high,low,vol,amount,datetime,code,date,date_stamp,time_stamp,type
datetime,,,,,,,,,,,,
2026-07-14 13:41:00,10.60,10.62,10.62,10.59,640300.0,6789860.0,2026-07-14 13:41:00,000001,2026-07-14,1.783958e+09,1.784008e+09,
2026-07-14 13:42:00,10.62,10.61,10.62,10.60,346300.0,3673281.0,2026-07-14 13:42:00,000001,2026-07-14,1.783958e+09,1.784008e+09,
2026-07-14 13:43:00,10.61,10.62,10.62,10.60,316800.0,3361279.0,2026-07-14 13:43:00,000001,2026-07-14,1.783958e+09,1.784008e+09,
2026-07-14 13:44:00,10.62,10.62,10.62,10.61,343300.0,3645397.0,2026-07-14 13:44:00,000001,2026-07-14,1.783958e+09,1.784008e+09,
2026-07-14 13:45:00,10.62,10.62,10.62,10.61,228700.0,2426970.0,2026-07-14 13:45:00,000001,2026-07-14,1.783958e+09,1.784008e+09,


In [42]:
pl_data

open,close,high,low,vol,amount,year,month,day,hour,minute,datetime
f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,str
10.6,10.62,10.62,10.59,640300.0,6.78986e6,2026,7,14,13,41,"""2026-07-14 13:…"
10.62,10.61,10.62,10.6,346300.0,3.673281e6,2026,7,14,13,42,"""2026-07-14 13:…"
10.61,10.62,10.62,10.6,316800.0,3.361279e6,2026,7,14,13,43,"""2026-07-14 13:…"
10.62,10.62,10.62,10.61,343300.0,3.645397e6,2026,7,14,13,44,"""2026-07-14 13:…"
10.62,10.62,10.62,10.61,228700.0,2.42697e6,2026,7,14,13,45,"""2026-07-14 13:…"
10.62,10.61,10.62,10.61,107100.0,1.136704e6,2026,7,14,13,46,"""2026-07-14 13:…"
10.62,10.63,10.63,10.61,408600.0,4.339149e6,2026,7,14,13,47,"""2026-07-14 13:…"
10.63,10.63,10.63,10.62,318600.0,3.384684e6,2026,7,14,13,48,"""2026-07-14 13:…"
10.63,10.63,10.64,10.62,655900.0,6.970896e6,2026,7,14,13,49,"""2026-07-14 13:…"


In [9]:
import polars as pl
from datetime import date

def QA_fetch_get_stock_min(code, start, end, frequence='1min', ip=None, port=None):
    ip, port = get_mainmarket_ip(ip, port)
    api = TdxHq_API()
    type_ = ''
    
    start_date = str(start)[:10]
    today_ = date.today()
    lens = QA_util_get_trade_gap(start_date, today_)
    
    # 频率映射与长度计算
    freq_map = {
        ('5', '5m', '5min', 'five'): (0, '5min', 48),
        ('1', '1m', '1min', 'one'): (8, '1min', 240),
        ('15', '15m', '15min', 'fifteen'): (1, '15min', 16),
        ('30', '30m', '30min', 'half'): (2, '30min', 8),
        ('60', '60m', '60min', '1h'): (3, '60min', 4),
    }
    
    str_freq = str(frequence)
    for keys, (f, t, multiplier) in freq_map.items():
        if str_freq in keys:
            frequence, type_, lens = f, t, multiplier * lens
            break
            
    if lens > 20800:
        lens = 20800
        
    # 分批获取数据并使用 polars 拼接
    frames = []
    with api.connect(ip, port):
        for i in range(int(lens / 800) + 1):
            offset = (int(lens / 800) - i) * 800
            df = api.to_df(api.get_security_bars(
                frequence, 
                _select_market_code(str(code)), 
                str(code), 
                offset, 
                800
            ))
            # 兼容 API 返回 Pandas 的情况
            if not isinstance(df, pl.DataFrame):
                df = pl.from_pandas(df)
            frames.append(df)
            
        data = pl.concat(frames, how="vertical")
        
    # 纯 Polars 原生表达式处理
    data = (
        data
        .drop(['year', 'month', 'day', 'hour', 'minute'])
        .with_columns([
            # 1. 字符串转时间 (同样加上 strict=False 防止数据源格式不统一报错)
            pl.col('datetime').str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False).alias('datetime'),
            # 2. 添加常量列
            pl.lit(str(code)).alias('code'),
            pl.lit(type_).alias('type'),
        ])
        # 3. 向量化生成 date, date_stamp, time_stamp
        .with_columns([
            pl.col('datetime').dt.strftime('%Y-%m-%d').alias('date'),
            pl.col('datetime').dt.truncate('1d').dt.epoch(time_unit='s').cast(pl.Float64).alias('date_stamp'),
            pl.col('datetime').dt.epoch(time_unit='s').cast(pl.Float64).alias('time_stamp'),
        ])
        # 4. 【修复点】：加上 strict=False，兼容 start/end 只有日期或包含时间的情况
        .filter(
            (pl.col('datetime') >= pl.lit(start).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False)) & 
            (pl.col('datetime') <= pl.lit(end).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S', strict=False))
        )
        # 5. 将 datetime 转回字符串（保持原函数返回格式一致）
        .with_columns(
            pl.col('datetime').dt.strftime('%Y-%m-%d %H:%M:%S').alias('datetime')
        )
    )
    
    return data

In [7]:
import polars as pl
from datetime import date

def QA_fetch_get_stock_min(code, start, end, frequence='1min', ip=None, port=None):
    ip, port = get_mainmarket_ip(ip, port)
    api = TdxHq_API()
    type_ = ''
    
    start_date = str(start)[:10]
    today_ = date.today()
    lens = QA_util_get_trade_gap(start_date, today_)
    
    # 频率映射与长度计算
    freq_map = {
        ('5', '5m', '5min', 'five'): (0, '5min', 48),
        ('1', '1m', '1min', 'one'): (8, '1min', 240),
        ('15', '15m', '15min', 'fifteen'): (1, '15min', 16),
        ('30', '30m', '30min', 'half'): (2, '30min', 8),
        ('60', '60m', '60min', '1h'): (3, '60min', 4),
    }
    
    str_freq = str(frequence)
    for keys, (f, t, multiplier) in freq_map.items():
        if str_freq in keys:
            frequence, type_, lens = f, t, multiplier * lens
            break
            
    if lens > 20800:
        lens = 20800
        
    # 分批获取数据并使用 polars 拼接
    frames = []
    with api.connect(ip, port):
        for i in range(int(lens / 800) + 1):
            offset = (int(lens / 800) - i) * 800
            df = api.to_df(api.get_security_bars(
                frequence, 
                _select_market_code(str(code)), 
                str(code), 
                offset, 
                800
            ))
            # 兼容 API 返回 Pandas 的情况
            if not isinstance(df, pl.DataFrame):
                df = pl.from_pandas(df)
            frames.append(df)
            
        data = pl.concat(frames, how="vertical")
        
    # 纯 Polars 原生表达式处理
    data = (
        data
        .drop(['year', 'month', 'day', 'hour', 'minute'])
        .with_columns([
            # 1. 字符串转时间
            pl.col('datetime').str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S').alias('datetime'),
            # 2. 添加常量列
            pl.lit(str(code)).alias('code'),
            pl.lit(type_).alias('type'),
        ])
        # 3. 向量化生成 date, date_stamp, time_stamp
        .with_columns([
            pl.col('datetime').dt.strftime('%Y-%m-%d').alias('date'),
            pl.col('datetime').dt.truncate('1d').dt.epoch(time_unit='s').cast(pl.Float64).alias('date_stamp'),
            pl.col('datetime').dt.epoch(time_unit='s').cast(pl.Float64).alias('time_stamp'),
        ])
        # 4. 【修复点】：将 start 和 end 也转换为 Datetime 类型后再进行比较
        .filter(
            (pl.col('datetime') >= pl.lit(start).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S')) & 
            (pl.col('datetime') <= pl.lit(end).str.strptime(pl.Datetime, fmt='%Y-%m-%d %H:%M:%S'))
        )
        # 5. 将 datetime 转回字符串（保持原函数返回格式一致）
        .with_columns(
            pl.col('datetime').dt.strftime('%Y-%m-%d %H:%M:%S').alias('datetime')
        )
    )
    
    return data